# Encrypt Google Drive Master Secrets

This notebook encrypts your `master-secrets.yaml` file stored in Google Drive
using **SOPS** and **age** — two free, open-source security tools.

**You do not need to understand the terminal or Ubuntu.  
Just run each cell from top to bottom using the ▶ button on the left.**

---

## What this notebook does

| Step | What happens |
|---|---|
| 1 | Connects to your Google Drive |
| 2 | Checks that `master-secrets.yaml` is present |
| 3 | Installs the encryption tools (SOPS and age) |
| 4 | Creates your personal encryption key (once only) |
| 5 | Encrypts `master-secrets.yaml` → `master-secrets.sops.yaml` |
| 6 | Verifies that the encrypted file is valid |
| 7 | Reminds you to delete the plaintext file manually |

---

## Before you start

Confirm all three boxes before running any cell:

- [ ] I created `My Drive / ICT_Bot_Secrets / master-secrets.yaml` in Google Drive
- [ ] I filled in all my real API keys and tokens in that file
- [ ] I have not shared that file with anyone

---
## Step 1 — Connect to Google Drive

Run the cell below. A pop-up will appear asking you to sign in to your Google
account and allow Colab to access your Drive.

This is safe — Colab only accesses your Drive during this session,
and only this notebook reads from your `ICT_Bot_Secrets` folder.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('Google Drive connected.')

---
## Step 2 — Check your secrets file

This cell confirms that `master-secrets.yaml` is in the expected folder.
If the file is missing, the cell will print clear instructions for what to do.

> **Default folder:** `My Drive / ICT_Bot_Secrets`  
> If you named your folder differently, change `SECRETS_DIR` in the cell below.

In [ ]:
import os, sys

# ── Change this only if you used a different folder name in Drive ──
SECRETS_DIR = '/content/drive/MyDrive/ICT_Bot_Secrets'
# ──────────────────────────────────────────────────────────────────

PLAINTEXT_FILE = os.path.join(SECRETS_DIR, 'master-secrets.yaml')
ENCRYPTED_FILE = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE   = os.path.join(SECRETS_DIR, 'age-keys.txt')

# Check the folder
if not os.path.isdir(SECRETS_DIR):
    print(f'ERROR: Folder not found:')
    print(f'  {SECRETS_DIR}')
    print()
    print('Steps to fix:')
    print('  1. Open Google Drive in your browser.')
    print('  2. Create a folder called  ICT_Bot_Secrets  inside My Drive.')
    print('  3. Run this cell again.')
    sys.exit(1)

# Check the plaintext file
if not os.path.isfile(PLAINTEXT_FILE):
    print('ERROR: Plaintext file not found:')
    print(f'  {PLAINTEXT_FILE}')
    print()
    print('Steps to fix:')
    print('  1. Open Google Drive in your browser.')
    print('  2. Go to  My Drive / ICT_Bot_Secrets.')
    print('  3. Make a copy of  ict-bot-master-secrets.template.yaml')
    print('     and rename the copy to  master-secrets.yaml')
    print('  4. Fill in your real API keys and tokens.')
    print('  5. Run this cell again.')
    sys.exit(1)

file_size = os.path.getsize(PLAINTEXT_FILE)
print(f'Found: {PLAINTEXT_FILE}')
print(f'  Size: {file_size} bytes')
print()
print('Ready to proceed to Step 3.')

---
## Step 3 — Install encryption tools

This cell downloads and installs two free, open-source tools:

- **age** — creates your personal encryption key and locks/unlocks data.
- **SOPS** (by the CNCF) — encrypts YAML files using your age key.

Nothing is sent to any third-party server — the tools are downloaded directly
from their official GitHub pages.

This step takes about 30–60 seconds. You will see progress messages below.

In [ ]:
import subprocess, sys

AGE_VERSION  = 'v1.1.1'
SOPS_VERSION = 'v3.8.1'

def shell(cmd, check=True):
    """Run a shell command and print its output. Exit on failure if check=True."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print('Error:', result.stderr.strip())
        if check:
            print('\nInstallation failed. Please try running this cell again.')
            sys.exit(result.returncode)
    return result

# ── Install age ──────────────────────────────────────────────────
print(f'Installing age {AGE_VERSION} ...')
shell(
    f'wget -q https://github.com/FiloSottile/age/releases/download/{AGE_VERSION}'
    f'/age-{AGE_VERSION}-linux-amd64.tar.gz -O /tmp/age.tar.gz'
)
shell('tar -xzf /tmp/age.tar.gz -C /tmp/')
shell('cp /tmp/age/age /tmp/age/age-keygen /usr/local/bin/')
shell('chmod +x /usr/local/bin/age /usr/local/bin/age-keygen')

age_check = shell('age --version', check=False)
if age_check.returncode != 0:
    print('ERROR: age did not install correctly.')
    sys.exit(1)
print(f'age installed: {age_check.stdout.strip()}')

# ── Install SOPS ─────────────────────────────────────────────────
print()
print(f'Installing SOPS {SOPS_VERSION} ...')
shell(
    f'wget -q https://github.com/getsops/sops/releases/download/{SOPS_VERSION}'
    f'/sops-{SOPS_VERSION}.linux.amd64 -O /usr/local/bin/sops'
)
shell('chmod +x /usr/local/bin/sops')

sops_check = shell('sops --version', check=False)
if sops_check.returncode != 0:
    print('ERROR: SOPS did not install correctly.')
    sys.exit(1)
print(f'SOPS installed: {sops_check.stdout.strip()}')

print()
print('Both tools are ready. Proceed to Step 4.')

---
## Step 4 — Create your age encryption key

An **age key** is a matched pair:

| Key | Purpose | Can I share it? |
|---|---|---|
| **Public key** | Locks (encrypts) the file | Yes — safe to share |
| **Private key** | Unlocks (decrypts) the file | **No — keep it secret** |

Both keys are saved together in:
`My Drive / ICT_Bot_Secrets / age-keys.txt`

**If that file already exists, this cell reuses it — your key is never overwritten.**

> ⚠️ **Back up `age-keys.txt` now.**  
> Copy it to a password manager or another safe location.  
> If you lose it, you cannot decrypt your secrets file.

In [ ]:
import subprocess, os, sys

SECRETS_DIR  = '/content/drive/MyDrive/ICT_Bot_Secrets'
AGE_KEY_FILE = os.path.join(SECRETS_DIR, 'age-keys.txt')

if os.path.isfile(AGE_KEY_FILE):
    print(f'age key file already exists:')
    print(f'  {AGE_KEY_FILE}')
    print('Reusing existing key — nothing was overwritten.')
else:
    print('No age key found. Generating a new one ...')
    result = subprocess.run(
        ['age-keygen', '-o', AGE_KEY_FILE],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('ERROR generating age key:')
        print(result.stderr)
        sys.exit(1)
    os.chmod(AGE_KEY_FILE, 0o600)  # owner read/write only
    print(f'New age key saved to:')
    print(f'  {AGE_KEY_FILE}')

# Read the PUBLIC key only — the private key is never printed
public_key = None
with open(AGE_KEY_FILE) as f:
    for line in f:
        if line.startswith('# public key:'):
            public_key = line.split(':', 1)[1].strip()
            break

if not public_key:
    print('ERROR: Could not read public key from age-keys.txt.')
    print('The file may be corrupt. Delete age-keys.txt and run this cell again.')
    sys.exit(1)

print()
print('Your age PUBLIC key (safe to share):')
print(f'  {public_key}')
print()
print('The PRIVATE key is stored only in age-keys.txt — it was not printed here.')
print()
print('Proceed to Step 5.')

---
## Step 5 — Encrypt your secrets file

This is the main step. SOPS will:

1. Read `master-secrets.yaml` from Drive.
2. Encrypt every value in the file using your age public key.
3. Write the result to `master-secrets.sops.yaml` in the same folder.

The encrypted file is safe to keep in Google Drive and safe to commit to GitHub.

**No secret values will be printed to screen during this step.**

In [ ]:
import subprocess, os, sys

SECRETS_DIR    = '/content/drive/MyDrive/ICT_Bot_Secrets'
PLAINTEXT_FILE = os.path.join(SECRETS_DIR, 'master-secrets.yaml')
ENCRYPTED_FILE = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE   = os.path.join(SECRETS_DIR, 'age-keys.txt')

# Confirm inputs exist
for path in (PLAINTEXT_FILE, AGE_KEY_FILE):
    if not os.path.isfile(path):
        print(f'ERROR: Required file not found: {path}')
        print('Please run the earlier steps before retrying this cell.')
        sys.exit(1)

# Re-read the public key from the key file
public_key = None
with open(AGE_KEY_FILE) as f:
    for line in f:
        if line.startswith('# public key:'):
            public_key = line.split(':', 1)[1].strip()
            break

if not public_key:
    print('ERROR: Could not read public key from age-keys.txt.')
    print('Run Step 4 again to regenerate the key.')
    sys.exit(1)

# Set the key file path so SOPS can find the private key for later decryption
env = os.environ.copy()
env['SOPS_AGE_KEY_FILE'] = AGE_KEY_FILE

print('Encrypting master-secrets.yaml ...')
print('(This may take a few seconds.)')

# stdout holds the encrypted content — it is written to disk, never printed
result = subprocess.run(
    [
        'sops', '--encrypt',
        '--age', public_key,
        '--input-type', 'yaml',
        '--output-type', 'yaml',
        PLAINTEXT_FILE,
    ],
    capture_output=True,
    env=env,
)

if result.returncode != 0:
    print('Encryption FAILED.')
    print('Error details:')
    print(result.stderr.decode())
    print()
    print('Do NOT delete master-secrets.yaml yet.')
    print('Try running Steps 3 and 4 again, then retry this cell.')
    sys.exit(1)

# Write the encrypted bytes to Drive
with open(ENCRYPTED_FILE, 'wb') as f:
    f.write(result.stdout)

enc_size = os.path.getsize(ENCRYPTED_FILE)
print()
print('Encryption succeeded.')
print(f'Encrypted file: {ENCRYPTED_FILE}')
print(f'  Size: {enc_size} bytes')
print()
print('The file contents were not printed. Your secrets are safe.')
print()
print('Proceed to Step 6.')

---
## Step 6 — Verify the encrypted file

Before you delete the plaintext file, this cell confirms two things:

1. `master-secrets.sops.yaml` exists in Drive and is non-empty.
2. SOPS can decrypt it using your age key — meaning no data was lost.

The decrypted content is **discarded immediately inside Colab** and never printed
to screen.

If this step prints `[OK]` for both checks, you are safe to delete the
plaintext file in Step 7.

In [ ]:
import subprocess, os, sys

SECRETS_DIR    = '/content/drive/MyDrive/ICT_Bot_Secrets'
ENCRYPTED_FILE = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE   = os.path.join(SECRETS_DIR, 'age-keys.txt')

all_ok = True

# ── Check 1: file exists and has content ─────────────────────────
if not os.path.isfile(ENCRYPTED_FILE):
    print('[FAIL] master-secrets.sops.yaml was not found.')
    print('       Run Step 5 again.')
    sys.exit(1)

enc_size = os.path.getsize(ENCRYPTED_FILE)
if enc_size == 0:
    print('[FAIL] master-secrets.sops.yaml is empty.')
    print('       Run Step 5 again.')
    sys.exit(1)

print(f'[OK]  master-secrets.sops.yaml exists ({enc_size} bytes)')

# ── Check 2: SOPS can decrypt it (output discarded, not printed) ──
env = os.environ.copy()
env['SOPS_AGE_KEY_FILE'] = AGE_KEY_FILE

result = subprocess.run(
    ['sops', '--decrypt', ENCRYPTED_FILE],
    capture_output=True,   # stdout and stderr captured; secrets never shown
    env=env,
)

if result.returncode != 0:
    print('[FAIL] Decryption test failed.')
    print('Error details:')
    print(result.stderr.decode())
    print()
    print('Do NOT delete master-secrets.yaml yet.')
    print('Something went wrong. Run Steps 4 and 5 again.')
    sys.exit(1)

print('[OK]  Decryption test passed — SOPS can read the encrypted file.')
print()
print('Your secrets are safely encrypted. No values were printed.')
print()
print('Proceed to Step 7.')

---
## Step 7 — Delete the plaintext file

Both checks in Step 6 passed. The encrypted file is ready.

The plaintext `master-secrets.yaml` should now be deleted so that your
real API keys are not sitting unprotected in Google Drive.

### How to delete it manually

1. Open [Google Drive](https://drive.google.com) in a new browser tab.
2. Navigate to **My Drive → ICT_Bot_Secrets**.
3. Right-click `master-secrets.yaml` → **Move to Trash**.
4. Click the Trash icon in the left sidebar → **Empty Trash**.

> **Why manually?**  
> We ask you to delete it yourself so you can visually confirm the encrypted
> file looks correct in Drive before removing the original.

---

## Final safety checklist

Before closing this notebook, confirm each item:

- [ ] `master-secrets.sops.yaml` is visible in **My Drive / ICT_Bot_Secrets** in Google Drive
- [ ] `age-keys.txt` is in the same folder
- [ ] I copied `age-keys.txt` to a password manager or another safe backup location
- [ ] I moved `master-secrets.yaml` to the Trash **and** emptied the Trash
- [ ] I did **not** share `age-keys.txt` with anyone
- [ ] I will disconnect the Colab session: **Runtime → Disconnect and delete runtime**

---

## What to do next

| Task | How |
|---|---|
| Use secrets on the VM | Copy `age-keys.txt` to the VM, then run `sops --decrypt master-secrets.sops.yaml` |
| Re-encrypt after updating keys | Fill a new `master-secrets.yaml` in Drive and run this notebook again |
| Generate lean `.env` files | Run `python scripts/generate_env.py` in the repo |

See `docs/claude/google-drive-master-secrets.md` in the repo for the full workflow.